# Inference and Ablations for LLaDA-Inspired BERT Diffusion

This notebook is focused on running iterative denoising generation and small ablation sweeps in Google Colab.

It assumes you want to use either the base `bert-base-uncased` model or a checkpoint you previously fine-tuned.

## 1. Install dependencies and clone the repo

In [ ]:
!pip -q install -U pip
!pip -q install -U transformers datasets torch tqdm

from pathlib import Path
import sys

REPO_URL = "https://github.com/lekkalapudiswetha-work/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM.git"
REPO_DIR = Path("/content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM")

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM
!pip -q install -e .

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

## 2. Imports and runtime config

In [ ]:
from dataclasses import dataclass
import json
import random

import numpy as np
import torch

from llada_bert import AblationRunner, BertMaskedLMWrapper, DiffusionNoiseScheduler, ExperimentConfig, IterativeDenoisingSampler


@dataclass
class InferenceConfig:
    model_name: str = "bert-base-uncased"
    checkpoint_path: str | None = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    prompt: str = "language models can become more useful when"
    sequence_length: int = 24
    steps: int = 12
    threshold: float = 0.85
    temperature: float = 0.9
    top_k: int = 25
    num_samples: int = 3
    remask_strategy: str = "low_confidence"


cfg = InferenceConfig()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

cfg

## 3. Choose which model to run

If you previously fine-tuned a model, set `cfg.checkpoint_path` to that saved directory. Otherwise the notebook will use `bert-base-uncased` directly.

In [ ]:
resolved_model_name = cfg.checkpoint_path or cfg.model_name
print("using model:", resolved_model_name)
print("device:", cfg.device)

## 4. Single inference run

In [ ]:
wrapper = BertMaskedLMWrapper(model_name=resolved_model_name, device=cfg.device)
scheduler = DiffusionNoiseScheduler(total_steps=cfg.steps, base_threshold=cfg.threshold)
sampler = IterativeDenoisingSampler(
    model=wrapper,
    scheduler=scheduler,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
)

result = sampler.sample(
    prompt=cfg.prompt,
    batch_size=cfg.num_samples,
    sequence_length=cfg.sequence_length,
    steps=cfg.steps,
    temperature=cfg.temperature,
    remask_strategy=cfg.remask_strategy,
)

for i, text in enumerate(result.texts, start=1):
    print(f"sample {i}: {text}")

## 5. Inspect convergence logs

In [ ]:
result.logger.as_rows()

## 6. Run a small ablation grid

This uses the repo's `AblationRunner` to compare different step counts and confidence thresholds.

In [ ]:
runner_config = ExperimentConfig(
    model_name=resolved_model_name,
    device=cfg.device,
    steps=cfg.steps,
    sequence_length=cfg.sequence_length,
    temperature=cfg.temperature,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
    seed=cfg.seed,
    num_samples=2,
    remask_strategy=cfg.remask_strategy,
)

runner = AblationRunner(runner_config)
ablation_grid = {
    "threshold": [0.75, 0.85, 0.92],
    "steps": [8, 12, 16],
}

ablation_results = runner.run_ablation_grid(cfg.prompt, ablation_grid)
len(ablation_results)

## 7. View compact ablation summaries

In [ ]:
compact_rows = []
for item in ablation_results:
    compact_rows.append({
        "threshold": item["config"]["threshold"],
        "steps": item["config"]["steps"],
        "final_mean_confidence": item["summary"].get("final_mean_confidence"),
        "final_masked_tokens": item["summary"].get("final_masked_tokens"),
        "sample_preview": item["texts"][0],
    })

compact_rows

## 8. Save results as JSON

In [ ]:
output_path = "/content/ablation_results.json"
with open(output_path, "w") as f:
    json.dump(ablation_results, f, indent=2)

print("saved:", output_path)

## 9. Suggested experiments

- compare base BERT against your fine-tuned checkpoint
- increase `sequence_length` and `steps`
- try lower temperature for more stable refinement
- compare `top_k` values such as 10, 25, and 50
- log multiple prompts and compare convergence traces